In [ ]:
pip install fsspec pandera numpy matplotlib pandas numpy pandas pyarrow aiohttp requests

In [ ]:
pip install pyarrow.parquet

In [1]:
from fpdf import FPDF
from datetime import datetime

In [2]:
import fsspec
print(fsspec.__version__)

2026.3.0


In [3]:
import numpy as np
import pandas as pd
import pyarrow as pa_arrow
import pyarrow.parquet as pq

In [4]:
url = 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINASC/csv/SINASC_2024_csv.zip'

#Definindo variáveis q serão utilizadas durante a execução do código.
arquivo_local = None
url_natalidade = url
n_amostra = 500_000

fonte = arquivo_local if (arquivo_local and Path(arquivo_local).exists()) else url_natalidade
print(f'Fonte configurada: {fonte[:80]}...' if len(fonte)>80 else f'Fonte: {fonte}')

Fonte configurada: https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINASC/csv/SINASC_2024_csv....


In [5]:
import requests
import zipfile
import io

print(f'{"_"*5}')
print("Baixando e descompactando o arquivo em memória")
print()
print(f'{"_"*50}')

_____
Baixando e descompactando o arquivo em memória

__________________________________________________


In [6]:
# Download do arquivo bruto - conteúdo binário

resposta_da_requisicao = requests.get(url_natalidade)
conteudo_do_arquivo = io.BytesIO(resposta_da_requisicao.content)

### 1. `resposta_da_requisicao = requests.get(url_natalidade)`

O `requests.get` é como se você estivesse digitando um endereço no seu navegador e apertando "Enter".
* **O que faz:** Envia uma requisição HTTP para a URL armazenada na variável `url_natalidade`.
* **O resultado:** Ele retorna um **objeto de resposta**. Esse objeto contém tudo o que veio do servidor: o conteúdo do arquivo, o código de status (se deu certo ou erro 404, por exemplo), os cabeçalhos, etc.

### 2. `conteudo_do_arquivo = io.BytesIO(resposta_da_requisicao.content)`

Esta linha é um "truque" de engenharia para tratar dados binários. Vamos dividir em duas partes:

* **`resposta_da_requisicao.content`**: Aqui você está acessando especificamente o "corpo" da resposta em formato de **bytes** (dados brutos). Se o arquivo for uma imagem ou um arquivo Excel, os dados estarão nesse formato binário.
* **`io.BytesIO(...)`**: O Python normalmente espera ler arquivos que estão gravados no disco (com um caminho tipo `C:/documentos/arquivo.csv`). O `BytesIO` cria um **arquivo virtual** na memória RAM.
* **O efeito prático:** Agora, a variável `conteudo_do_arquivo` se comporta exatamente como se fosse um arquivo aberto no seu computador.

---

### Por que usar isso?

Imagine que você está baixando uma planilha do IBGE. Em vez de fazer isso:
1. Baixar o arquivo.
2. Salvar em `C:/downloads/dados.xlsx`.
3. Abrir o arquivo com o Pandas.
4. Deletar o arquivo depois.

Você faz tudo direto na memória, o que é **mais rápido** e **mais limpo**, pois não deixa "lixo" (arquivos temporários) na sua máquina.

> **Dica:** Geralmente, após essas linhas, você usaria algo como `pandas.read_excel(conteudo_do_arquivo)` para começar a analisar os dados de natalidade.


In [7]:
# "Des"compactação e leitura

with zipfile.ZipFile(conteudo_do_arquivo) as zippado:
    arquivos_internos = zippado.namelist()
    with zippado.open(arquivos_internos[0]) as f:
        df_natalidade2024 = pd.read_csv(f, sep=';', nrows=n_amostra, encoding='latin1')

### 1. `with zipfile.ZipFile(conteudo_do_arquivo) as zippado:`
* **O que faz:** O comando `ZipFile` lê os dados binários que você baixou (que estão na variável `conteudo_do_arquivo`) e os interpreta como um arquivo compactado .zip.
* **O `with`:** É uma boa prática em Python chamada "Gerenciador de Contexto". Ele garante que, assim que você terminar de ler o conteúdo, o arquivo seja "fechado" na memória, liberando espaço.

### 2. `arquivos_internos = zippado.namelist()`
* **O que faz:** Um arquivo ZIP pode ter vários arquivos dentro dele (planilhas, textos, fotos). O `namelist()` cria uma lista com os nomes de tudo o que está lá dentro.
* **Exemplo:** Se dentro do ZIP tiver um arquivo chamado `natalidade_2024.csv`, essa lista será `['natalidade_2024.csv']`.

### 3. `with zippado.open(arquivos_internos[0]) as f:`
* **O que faz:** Ele acessa o primeiro arquivo da lista (`[0]`) e o "abre" para leitura. A variável `f` passa a ser o fluxo de dados desse arquivo específico que estava escondido dentro do ZIP.

### 4. `df = pd.read_csv(f, sep=';', ...)`
Aqui é onde a mágica acontece e os dados viram uma tabela (DataFrame) do Pandas:
* **`f`**: O Pandas lê diretamente do arquivo aberto na memória.
* **`sep=';'`**: Avisa que as colunas no arquivo CSV são separadas por ponto e vírgula.
* **`nrows=n_amostra`**: Comando muito útil! Ele diz ao Python: "Não leia o arquivo inteiro (que pode ter milhões de linhas), leia apenas as primeiras `X` linhas". Isso economiza muita memória.
* **`encoding='latin1'`**: Define o padrão de caracteres. No Brasil, arquivos de órgãos governamentais costumam usar `latin1` para que acentos e cedilhas não fiquem bugados.

---

### Resumo do Fluxo


1. **`ZipFile`**: Identifica o "pacote".
2. **`namelist`**: Olha o que tem na "etiqueta" do pacote.
3. **`open`**: Abre um item específico do pacote.
4. **`read_csv`**: Transforma o texto desse item em uma tabela organizada.

In [8]:
# Validação da Compressão

compressao = "ZIP" if fonte.endswith('.zip') else "Nenhuma"

In [33]:
uso_memoria = df_natalidade2024.memory_usage(deep=True) / 1024

print(f'{"Coluna":<21}|{"Tipo":<12}|{"RAM":<5}')
print(f'{"_"*50}')
for meta in df_natalidade2024.columns:
    tipo = str(df_natalidade2024[meta].dtype)
    espaco = uso_memoria[meta]
    print(f' {meta:<20}| {tipo:<10} |{espaco:>12.1f}')
print(f'{"_"*50}')

Coluna               |Tipo        |RAM  
__________________________________________________
 contador            | int64      |      3906.2
 ORIGEM              | int64      |      3906.2
 CODESTAB            | float64    |      3906.2
 CODMUNNASC          | int64      |      3906.2
 LOCNASC             | int64      |      3906.2
 IDADEMAE            | float64    |      3906.2
 ESTCIVMAE           | float64    |      3906.2
 ESCMAE              | float64    |      3906.2
 CODOCUPMAE          | float64    |      3906.2
 QTDFILVIVO          | float64    |      3906.2
 QTDFILMORT          | float64    |      3906.2
 CODMUNRES           | int64      |      3906.2
 GESTACAO            | float64    |      3906.2
 GRAVIDEZ            | float64    |      3906.2
 PARTO               | float64    |      3906.2
 CONSULTAS           | float64    |      3906.2
 DTNASC              | int64      |      3906.2
 HORANASC            | float64    |      3906.2
 SEXO                | int64      |      390

In [34]:
df_natalidade2024.head()

,contador,ORIGEM,CODESTAB,CODMUNNASC,LOCNASC,IDADEMAE,ESTCIVMAE,ESCMAE,CODOCUPMAE,QTDFILVIVO,...,TPFUNCRESP,TPDOCRESP,DTDECLARAC,ESCMAEAGR1,STDNEPIDEM,STDNNOVA,CODPAISRES,TPROBSON,PARIDADE,KOTELCHUCK
0,1,1,2516500.0,110001,1,24.0,1.0,3.0,999992.0,1.0,...,2.0,3.0,14022024.0,3.0,0.0,1,1.0,5,1,2
1,2,1,2516500.0,110001,1,29.0,2.0,5.0,999992.0,0.0,...,2.0,3.0,17042024.0,8.0,0.0,1,1.0,2,0,2
2,3,1,2516500.0,110001,1,30.0,5.0,4.0,622020.0,2.0,...,2.0,3.0,29052024.0,5.0,0.0,1,1.0,5,1,5
3,4,1,2516500.0,110001,1,14.0,5.0,4.0,999991.0,0.0,...,2.0,3.0,27052024.0,4.0,0.0,1,1.0,1,0,5
4,5,1,2516500.0,110001,1,24.0,2.0,4.0,999992.0,1.0,...,2.0,3.0,13052024.0,5.0,0.0,1,1.0,5,1,5


Aqui está a lista organizada por categorias para facilitar o seu entendimento:

### 1. Informações de Identificação e Registro
* **contador:** Um número sequencial para controle interno do arquivo.
* **ORIGEM:** Indica se o dado veio de uma base estadual, municipal ou federal.
* **CODESTAB:** Código do Cadastro Nacional de Estabelecimentos de Saúde (CNES) onde ocorreu o nascimento.
* **DTCADASTRO / DTRECEBIM / DTRECORIGA:** Datas de cadastro, recebimento e registro original no sistema.
* **NUMEROLOTE:** Número do lote de processamento dos dados.
* **VERSAOSIST:** Versão do sistema de software utilizada para coletar os dados.

### 2. Localização Geográfica
* **CODMUNNASC:** Código do município onde a criança nasceu (padrão IBGE).
* **LOCNASC:** Tipo de local do nascimento (1 para Hospital, 2 para outro estabelecimento de saúde, 3 para Domicílio, etc.).
* **CODMUNRES:** Código do município onde a mãe reside.
* **CODMUNNATU / CODUFNATU:** Município e Unidade da Federação de naturalidade da mãe.
* **CODPAISRES:** Código do país de residência (geralmente Brasil).

### 3. Características da Mãe e do Pai
* **IDADEMAE / IDADEPAI:** Idade da mãe e do pai em anos.
* **ESTCIVMAE:** Estado civil da mãe (Solteira, casada, viúva, etc.).
* **ESCMAE / ESCMAE2010 / ESCMAEAGR1:** Escolaridade da mãe (existem diferentes colunas por conta de mudanças metodológicas ao longo dos anos).
* **SERIESCMAE:** Série escolar que a mãe cursava/concluiu.
* **CODOCUPMAE:** Código da ocupação da mãe (baseado na CBO - Classificação Brasileira de Ocupações).
* **RACACORMAE:** Raça ou cor da mãe.
* **DTNASCMAE:** Data de nascimento da mãe.
* **NATURALMAE:** Nacionalidade da mãe.

### 4. Histórico Reprodutivo (Paridade)
* **QTDFILVIVO / QTDFILMORT:** Quantidade de filhos vivos e mortos em gestações anteriores.
* **QTDGESTANT:** Número total de gestações anteriores.
* **QTDPARTNOR / QTDPARTCES:** Quantidade de partos normais e cesáreas anteriores.
* **PARIDADE:** Indica se a mãe é primípara (primeiro filho) ou multipara.

### 5. Detalhes da Gestação e Parto
* **GESTACAO:** Semanas de gestação (em faixas, ex: 37 a 41 semanas).
* **SEMAGESTAC:** Número exato de semanas de gestação.
* **TPMETESTIM:** Método utilizado para estimar a idade gestacional (ex: Exame físico ou Ultrassom).
* **GRAVIDEZ:** Tipo de gravidez (Única, dupla, tripla ou mais).
* **PARTO:** Tipo de parto (Vaginal ou Cesáreo).
* **CONSULTAS / CONSPRENAT:** Número de consultas de pré-natal realizadas.
* **MESPRENAT:** Em qual mês de gestação a mãe iniciou o pré-natal.
* **DTULTMENST:** Data da última menstruação.
* **STTRABPART:** Indica se a mãe entrou em trabalho de parto.
* **STCESPARTO:** Indica se a cesárea ocorreu antes do trabalho de parto iniciar.
* **TPAPRESENT:** Tipo de apresentação do bebê (Cefálica, pélvica, etc.).

### 6. Características do Recém-Nascido
* **DTNASC / HORANASC:** Data e hora exata do nascimento.
* **SEXO:** Sexo do bebê (Masculino ou Feminino).
* **APGAR1 / APGAR5:** Notas de Apgar no 1º e 5º minuto (avalia a vitalidade do bebê).
* **RACACOR:** Raça ou cor do recém-nascido.
* **PESO:** Peso ao nascer em gramas.
* **IDANOMAL / CODANOMAL:** Indica se foi detectada alguma anomalia congênita e qual o código (CID-10).

### 7. Indicadores e Classificações Técnicas
* **TPROBSON:** Classificação de Robson (agrupa as mulheres em 10 grupos para analisar taxas de cesárea).
* **KOTELCHUCK:** Índice que mede a adequação do pré-natal (combina o início do pré-natal com o número de consultas).
* **OPORT_DN:** Indicador de oportunidade de registro do nascimento.
* **DIFDATA:** Diferença de dias entre o nascimento e o registro.
* **TPNASCASSI:** Tipo de assistência no nascimento (Médico, enfermeira, parteira, etc.).
* **TPFUNCRESP / TPDOCRESP:** Tipo de função e documento do responsável pelo preenchimento da Declaração de Nascido Vivo.

---

**Uma dica rápida:** Se você estiver analisando esses dados em Python ou R, note que muitas colunas estão como `float64`. Isso geralmente acontece porque existem valores faltantes (`NaN`) na base, o que faz com que o sistema entenda os números inteiros como decimais.

In [35]:
lista_colunas_principais = [
    #PAIS
    'IDADEMAE',
    'IDADEPAI',

    #GESTAÇÃO
    'SEMAGESTAC', #SEMANAS DE GESTAÇÃO
    'QTDPARTNOR', #QUANTIDADE DE PARTOS NORMAIS
    'QTDPARTCES', #QUANTIDADE DE PARTOS CESÁRIOS
    'CONSPRENAT', #QUANTIDADE DE PRENATAIS FEITOS.
    'KOTELCHUCK', #INDICE QUE CLASSIFICA A QUALIDADE DO PRENATAL FEITO
    'DIFDATA', #Diferença de dias entre o nascimento e o registro.
    'GRAVIDEZ', #INDICA SE A GRAVIDEZ FOI SIMPLES, DUPLA, TRIPLA E ETC.

    #BEBÊ E NASCIMENTO
    'PESO',
    'HORANASC',
    'DTNASC',
    'SEXO'
]

In [37]:
type(lista_colunas_principais)

list

In [38]:
#organizando um novo dataframe

df_final = df_natalidade2024[lista_colunas_principais].copy()

In [39]:
df_final.head()

,IDADEMAE,IDADEPAI,SEMAGESTAC,QTDPARTNOR,QTDPARTCES,CONSPRENAT,KOTELCHUCK,DIFDATA,GRAVIDEZ,PESO,HORANASC,DTNASC,SEXO
0,24.0,NaN,38.0,0.0,1.0,2.0,2,20,1.0,3120.0,845.0,14022024,1
1,29.0,41.0,39.0,0.0,0.0,8.0,2,22,1.0,3564.0,850.0,17042024,1
2,30.0,35.0,39.0,0.0,2.0,8.0,5,5,1.0,2816.0,1006.0,29052024,1
3,14.0,17.0,38.0,0.0,0.0,10.0,5,7,1.0,3126.0,900.0,27052024,1
4,24.0,29.0,39.0,0.0,1.0,11.0,5,21,1.0,3622.0,835.0,13052024,1


In [40]:
df_final.info(memory_usage=True)

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   IDADEMAE    499988 non-null  float64
 1   IDADEPAI    83432 non-null   float64
 2   SEMAGESTAC  493614 non-null  float64
 3   QTDPARTNOR  482794 non-null  float64
 4   QTDPARTCES  480498 non-null  float64
 5   CONSPRENAT  489859 non-null  float64
 6   KOTELCHUCK  500000 non-null  int64  
 7   DIFDATA     500000 non-null  int64  
 8   GRAVIDEZ    499560 non-null  float64
 9   PESO        499889 non-null  float64
 10  HORANASC    499726 non-null  float64
 11  DTNASC      500000 non-null  int64  
 12  SEXO        500000 non-null  int64  
dtypes: float64(9), int64(4)
memory usage: 49.6 MB


In [41]:
df_final.to_parquet('sinasc2024_reduzidissima_final.parquet', index=False)